# HH-PC SNN — Ablation Study on SHD (20 classes)

Same ablation framework as the MNIST/N-MNIST/Caltech study, run on the SHD
pipeline (speaker-level split, `tau_ref=0.15`, per-layer `I_bias`, soft PC targets).

**Table A — core ablations**
| factor | levels |
|---|---|
| neuron model | HH / LIF / Analog |
| inference mode | PC / FF |
| input encoding | Poisson / latency-first |

**Table B — SHD-specific ablations**
| factor | levels |
|---|---|
| PC target | soft (0.9 / 0.005) vs hard one-hot |
| augmentation | on vs off |
| refractory | `tau_ref=0.15` (5 steps) vs `2.0` (67 steps, > `steps_spk`) |

Reports Acc, macro-F1, precision, recall, PC energy, spike rate, spikes/sample, macro-OvR AUC, train time.


In [3]:
# ── Cell 1: imports + config ──────────────────────────────────────────────
import os, sys, glob, gzip, shutil, time, json, csv, math, copy, types, urllib.request
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Subset, Dataset

_hits = (glob.glob('/kaggle/input/**/hhpc_base.py', recursive=True) +
         glob.glob('/kaggle/working/**/hhpc_base.py', recursive=True) +
         glob.glob('./hhpc_base.py'))
assert _hits, 'hhpc_base.py not found — attach the dataset'
sys.path.insert(0, os.path.dirname(os.path.abspath(_hits[0])))
from hhpc_base import (PCSNNet, HHNeuron, MetricAccumulator, EarlyStopper,
                       one_hot, set_seed, default_device)
print('hhpc_base at:', _hits[0])

# ---------------- CONFIG (identical to the SHD main run unless ablated) ----
DATA_DIR    = '/kaggle/working/shd'
OUT_DIR     = '/kaggle/working/ablation_results'
N_CH, POOL, TBINS, T_MAX = 700, 5, 50, 1.0
NUM_CLASSES = 20
CH   = N_CH // POOL
D_IN = TBINS * CH

HIDDEN      = 400
DT          = 0.03
TAU_REF     = 0.15
I_BIAS      = [1.5, 3.0]      # [hidden, output]
CURRENT_GAIN= 30.0
THR         = 0.8
LIF_TAU     = 0.3
STEPS_SPK   = 50
T_TRAIN     = 100
T_EVAL      = 50
ETA_X       = 0.05
LR          = 2e-4
WD          = 2e-4
EPOCHS      = 25              # lowered from 40: 9 variants per full sweep
PATIENCE    = 8
BATCH       = 128
N_VAL_SPK   = 2
SEED        = 123

TGT_HI, TGT_LO = 0.9, 0.1     # soft target: hi on true class, lo/C elsewhere
TIME_SHIFT_MAX, CH_DROP_P = 4, 0.15

DEVICE = default_device()
set_seed(SEED)
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)
print('device:', DEVICE, '| D_in:', D_IN, '| classes:', NUM_CLASSES)


hhpc_base at: /kaggle/input/datasets/ishraqkamaladib/hhpc-base/hhpc_base.py
device: cuda | D_in: 7000 | classes: 20


## 1. Data

In [4]:
# ── Cell 2: locate or download shd_train.h5 / shd_test.h5 ─────────────────
try:
    import h5py
except ImportError:
    os.system('pip install -q h5py'); import h5py

URLS = {'train': 'https://zenkelab.org/datasets/shd_train.h5.gz',
        'test':  'https://zenkelab.org/datasets/shd_test.h5.gz'}

def get_h5(split):
    for pat in (f'/kaggle/input/**/shd_{split}.h5', f'{DATA_DIR}/shd_{split}.h5',
                f'/kaggle/input/**/shd_{split}.h5.gz'):
        hits = glob.glob(pat, recursive=True)
        if hits:
            p = hits[0]
            if p.endswith('.gz'):
                out = f'{DATA_DIR}/shd_{split}.h5'
                if not os.path.exists(out):
                    with gzip.open(p, 'rb') as fi, open(out, 'wb') as fo:
                        shutil.copyfileobj(fi, fo)
                return out
            return p
    gz, out = f'{DATA_DIR}/shd_{split}.h5.gz', f'{DATA_DIR}/shd_{split}.h5'
    print(f'downloading {split} ...', flush=True)
    urllib.request.urlretrieve(URLS[split], gz)
    with gzip.open(gz, 'rb') as fi, open(out, 'wb') as fo:
        shutil.copyfileobj(fi, fo)
    return out

H5_TRAIN, H5_TEST = get_h5('train'), get_h5('test')
print('ok:', H5_TRAIN, H5_TEST)


downloading train ...
downloading test ...
ok: /kaggle/working/shd/shd_train.h5 /kaggle/working/shd/shd_test.h5


In [5]:
# ── Cell 3: bin (times, units) -> TBINS x CH frame ────────────────────────
def load_shd(path, tbins=TBINS, pool=POOL, n_ch=N_CH, t_max=T_MAX):
    ch = n_ch // pool
    with h5py.File(path, 'r') as f:
        times, units = f['spikes']['times'], f['spikes']['units']
        labels = np.asarray(f['labels'], np.int64)
        spk = (np.asarray(f['extra']['speaker'], np.int64)
               if 'extra' in f and 'speaker' in f['extra'] else np.zeros(len(labels), np.int64))
        X = np.zeros((len(labels), tbins * ch), np.float32)
        for i in range(len(labels)):
            t = np.asarray(times[i], np.float64); u = np.asarray(units[i], np.int64)
            if t.size == 0:
                continue
            ti = np.clip((t / t_max * tbins).astype(np.int64), 0, tbins - 1)
            ui = np.clip(u // pool, 0, ch - 1)
            cnt = np.log1p(np.bincount(ti * ch + ui, minlength=tbins * ch).astype(np.float32))
            m = cnt.max(); X[i] = cnt / m if m > 0 else cnt
    return X, labels, spk

t0 = time.time()
Xtr_all, ytr_all, spk_tr = load_shd(H5_TRAIN)
Xte,     yte,     spk_te = load_shd(H5_TEST)
print(f'loaded in {time.time()-t0:.1f}s | train {Xtr_all.shape} test {Xte.shape}')


loaded in 5.9s | train (8156, 7000) test (2264, 7000)


In [6]:
# ── Cell 4: speaker-level split + loader factory (augmentation is ablatable) ──
rng = np.random.RandomState(SEED)
spk_ids = np.unique(spk_tr)
val_spk = rng.choice(spk_ids, size=min(N_VAL_SPK, len(spk_ids)), replace=False)
val_mask = np.isin(spk_tr, val_spk)
tr_idx, val_idx = np.where(~val_mask)[0], np.where(val_mask)[0]
print('held-out val speakers:', val_spk, '| val n =', len(val_idx))

class AugmentedSHD(Dataset):
    """Random circular time shift + pooled-channel dropout, applied per __getitem__."""
    def __init__(self, X, y, tbins=TBINS, ch=CH, max_shift=TIME_SHIFT_MAX, drop_p=CH_DROP_P):
        self.X, self.y, self.tbins, self.ch = X, y, tbins, ch
        self.max_shift, self.drop_p = max_shift, drop_p
    def __len__(self):  return len(self.y)
    def __getitem__(self, i):
        x = self.X[i].reshape(self.tbins, self.ch)
        s = np.random.randint(-self.max_shift, self.max_shift + 1)
        if s: x = np.roll(x, s, axis=0)
        if self.drop_p > 0:
            x = x * (np.random.rand(self.ch) >= self.drop_p)[None, :]
        return torch.from_numpy(np.ascontiguousarray(x.reshape(-1))), int(self.y[i])

def to_ds(X, y, idx=None):
    if idx is not None: X, y = X[idx], y[idx]
    return TensorDataset(torch.from_numpy(X), torch.from_numpy(y))

_KW = dict(num_workers=0, pin_memory=DEVICE.startswith('cuda'))

def make_loaders(augment=True):
    """Returns (train, val, test). val/test are never augmented."""
    tr_ds = (AugmentedSHD(Xtr_all[tr_idx], ytr_all[tr_idx]) if augment
             else to_ds(Xtr_all, ytr_all, tr_idx))
    return (DataLoader(tr_ds, BATCH, shuffle=True, **_KW),
            DataLoader(to_ds(Xtr_all, ytr_all, val_idx), BATCH, shuffle=False, **_KW),
            DataLoader(to_ds(Xte, yte), BATCH, shuffle=False, **_KW))

TR_AUG, VA, TE = make_loaders(augment=True)
TR_RAW, _, _   = make_loaders(augment=False)
print(len(TR_AUG.dataset), len(VA.dataset), len(TE.dataset))


held-out val speakers: [6 0] | val n = 1360
6796 1360 2264


## 2. Neuron variants + model builder

In [7]:
# ── Cell 5: LIF neuron + generic patched forward_proxies ──────────────────
class LIFNeuron(nn.Module):
    def __init__(self, N, dt=DT, device='cpu', thr=THR, reset=0.0, tau=LIF_TAU, tau_ref=TAU_REF):
        super().__init__()
        self.N, self.dt, self.device = N, float(dt), device
        self.thr, self.reset, self.tau = float(thr), float(reset), float(tau)
        self.refr_steps = max(1, int(round(tau_ref / dt)))
        self.reset_states(1)
    def reset_states(self, B):
        self.B = B
        self.Vm   = torch.zeros(B, self.N, device=self.device)
        self.refr = torch.zeros(B, self.N, device=self.device)
    def forward(self, I):
        if I.shape[0] != self.B: self.reset_states(I.shape[0])
        can = (self.refr <= 0)
        Vn  = self.Vm + (-self.Vm + I) * (self.dt / max(self.tau, 1e-6))
        spk = ((Vn >= self.thr) & can).float()
        self.Vm = torch.where(spk.bool(), torch.full_like(Vn, self.reset), Vn)
        self.refr = torch.where(spk.bool(), torch.full_like(self.refr, float(self.refr_steps)),
                                (self.refr - 1.).clamp(min=0.))
        return spk, self.Vm


def _forward_proxies(self, x_in, steps_spk):
    """Handles neuron_type in {hh, lif, analog} x encoding in {poisson, latency_first},
    with per-layer I_bias. Signature-compatible with PCSNNet.forward_proxies."""
    x0 = x_in.to(self.device).clamp(0, 1); B = x0.size(0)

    if self.neuron_type == 'analog':                      # non-spiking sigmoid proxy
        proxies, r = [x0], x0
        for i in range(self.L):
            r = torch.sigmoid(F.linear(r, self.syn[i].weight, self.syn[i].bias))
            proxies.append(r.clamp(0, 1))
        self._last_spike_sums = None
        self._input_spikes = 0.
        return proxies

    for cell in self.cells:
        cell.reset_states(B)

    if self.input_encoding == 'latency_first':
        lat = (steps_spk * (1.0 - x0)).clamp(0., float(steps_spk))
        tgrid = torch.arange(1, steps_spk + 1, device=self.device).view(1, 1, -1)
        spk_train = (lat.unsqueeze(-1) <= tgrid).float()
        fired = torch.zeros_like(x0, dtype=torch.bool)
    else:
        p = (x0 * self.poisson_scale).clamp(0, 1)
        spk_train = (torch.rand(B, x0.size(1), steps_spk, device=self.device)
                     < p.unsqueeze(-1)).float()
        fired = None

    spike_sums = [torch.zeros(B, self.sizes[i+1], device=self.device) for i in range(self.S)]
    in_spk = 0.
    for t in range(steps_spk):
        if self.input_encoding == 'poisson':
            r = spk_train[:, :, t]
        else:                                             # first-spike only
            r = (spk_train[:, :, t] * (~fired)).float(); fired.logical_or_(r.bool())
        in_spk += float(r.sum())
        for i in range(self.S):
            I = F.linear(r, self.syn[i].weight, self.syn[i].bias) * self.current_gain \
                + self.I_bias_list[i]
            spk, _ = self.cells[i](I)
            spike_sums[i] += spk
            r = spk
    self._last_spike_sums = spike_sums
    self._input_spikes = in_spk
    return [x0] + [(ss / float(steps_spk)).clamp(0, 1) for ss in spike_sums]


def build_model(neuron_type='hh', encoding='poisson', tau_ref=TAU_REF,
                i_bias=I_BIAS, hidden=HIDDEN, seed=SEED):
    set_seed(seed)
    m = PCSNNet(layer_sizes=[D_IN, hidden, NUM_CLASSES], dt=DT, device=DEVICE,
                current_gain=CURRENT_GAIN, thr=THR, lr=LR, weight_decay=WD,
                pc_activation='relu', input_encoding=encoding).to(DEVICE)
    m.neuron_type = neuron_type
    m.input_encoding = encoding
    m.I_bias_list = list(i_bias)

    if neuron_type == 'lif':
        m.cells = nn.ModuleList([LIFNeuron(m.sizes[i+1], dt=DT, device=DEVICE,
                                           thr=THR, tau=LIF_TAU, tau_ref=tau_ref)
                                 for i in range(m.S)]).to(DEVICE)
    elif neuron_type == 'hh':
        for c in m.cells:
            c.refr_steps = max(1, int(round(tau_ref / DT)))
    else:                                                  # analog: cells unused
        pass

    m.forward_proxies = types.MethodType(_forward_proxies, m)
    m._input_spikes = 0.
    return m


## 3. Targets, evaluation, training

In [8]:
# ── Cell 6: soft targets + evaluator + spike accounting ───────────────────
def soft_one_hot(y, C, hi=TGT_HI, lo=TGT_LO):
    t = torch.full((y.size(0), C), lo / C, device=y.device)
    t.scatter_(1, y.view(-1, 1).long(), hi)
    return t

def make_target(y, C, soft=True):
    return soft_one_hot(y, C) if soft else one_hot(y, C)


@torch.no_grad()
def evaluate(model, loader, eval_mode='pc', T_infer=T_EVAL, eval_seed=1234, want_scores=False):
    model.eval()
    Cn = model.sizes[-1]
    acc_pc, acc_ff = MetricAccumulator(Cn), MetricAccumulator(Cn)
    tot_e, tot_n = 0.0, 0
    scores, labels = [], []
    with torch.random.fork_rng():
        torch.manual_seed(eval_seed)
        for x, y in loader:
            x = x.to(DEVICE).view(x.size(0), -1); y = y.to(DEVICE); B = x.size(0)
            proxies = model.forward_proxies(x, STEPS_SPK)
            acc_ff.update(proxies[-1].argmax(1).cpu(), y.cpu())
            if eval_mode == 'pc':
                xs, _, _, _ = model.pc_infer(proxies, None, T_infer, ETA_X, False)
                out = xs[-1]
            else:
                out = proxies[-1]
            acc_pc.update(out.argmax(1).cpu(), y.cpu())
            if want_scores:
                scores.append(out.cpu().numpy()); labels.append(y.cpu().numpy())
            _, _, _, e = model.pc_infer(proxies, one_hot(y, Cn), T_infer, ETA_X, True)
            tot_e += e * B; tot_n += B
    m, mff = acc_pc.compute(), acc_ff.compute()
    r = {'acc': m['acc'], 'f1': m['f1'], 'precision': m['precision'], 'recall': m['recall'],
         'ff_acc': mff['acc'], 'loss': tot_e / max(tot_n, 1)}
    if want_scores:
        r['scores'] = np.concatenate(scores); r['labels'] = np.concatenate(labels)
    return r


@torch.no_grad()
def spike_stats(model, loader, eval_seed=1234):
    if model.neuron_type == 'analog':
        return {'per_layer': [], 'total': 0.0, 'spikes_per_sample': 0.0, 'input_per_sample': 0.0}
    model.eval()
    S = model.S
    tot_spk, tot_den, n, in_spk = [0.]*S, [0.]*S, 0, 0.
    with torch.random.fork_rng():
        torch.manual_seed(eval_seed)
        for x, _ in loader:
            x = x.to(DEVICE).view(x.size(0), -1); B = x.size(0)
            model.forward_proxies(x, STEPS_SPK)
            in_spk += model._input_spikes
            for li, ss in enumerate(model.last_spike_sums()):
                tot_spk[li] += float(ss.sum())
                tot_den[li] += float(B * ss.shape[1] * STEPS_SPK)
            n += B
    return {'per_layer': [tot_spk[i]/max(tot_den[i],1.) for i in range(S)],
            'total': sum(tot_spk)/max(sum(tot_den),1.),
            'spikes_per_sample': sum(tot_spk)/max(n,1),
            'input_per_sample': in_spk/max(n,1)}


def macro_auc(scores, labels, C=NUM_CLASSES):
    from sklearn.metrics import roc_auc_score
    from sklearn.preprocessing import label_binarize
    try:
        return float(roc_auc_score(label_binarize(labels, classes=np.arange(C)),
                                   scores, average='macro'))
    except Exception:
        return float('nan')


In [9]:
# ── Cell 7: single-variant runner ─────────────────────────────────────────
def run_variant(name, neuron='hh', encoding='poisson', eval_mode='pc',
                soft=True, augment=True, tau_ref=TAU_REF, i_bias=I_BIAS,
                epochs=EPOCHS, patience=PATIENCE, verbose=False):
    tr = TR_AUG if augment else TR_RAW
    model = build_model(neuron, encoding, tau_ref, i_bias)
    ckpt = os.path.join(OUT_DIR, 'ckpt_' + name.replace(' ', '_').replace('/', '_') + '.pt')

    stopper = EarlyStopper(patience=patience)
    hist = []
    t0 = time.time()
    for ep in range(1, epochs + 1):
        model.train()
        for x, y in tr:
            x = x.to(DEVICE).view(x.size(0), -1); y = y.to(DEVICE)
            model.train_step(x, make_target(y, NUM_CLASSES, soft), STEPS_SPK, T_TRAIN, ETA_X)
        va = evaluate(model, VA, eval_mode)
        hist.append({'epoch': ep, 'val_acc': va['acc'], 'val_f1': va['f1'], 'val_loss': va['loss']})
        stop, improved = stopper.step(va['f1'], ep)
        if improved:
            torch.save(model.state_dict(), ckpt)
        if verbose:
            print(f"    ep {ep:02d} val_acc {va['acc']*100:5.2f} val_F1 {va['f1']*100:5.2f}"
                  f"{'  *' if improved else ''}", flush=True)
        if stop:
            if verbose: print(f'    early stop @ {ep} (best {stopper.best_epoch})')
            break
    elapsed = time.time() - t0

    if os.path.exists(ckpt):
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    te  = evaluate(model, TE, eval_mode, want_scores=True)
    spk = spike_stats(model, TE)

    row = {'variant': name, 'neuron': neuron.upper(), 'encoding': encoding,
           'inference': eval_mode.upper(), 'targets': 'soft' if soft else 'hard',
           'augment': int(augment), 'tau_ref': tau_ref,
           'acc': round(te['acc']*100, 2), 'f1': round(te['f1']*100, 2),
           'precision': round(te['precision']*100, 2), 'recall': round(te['recall']*100, 2),
           'auc': round(macro_auc(te['scores'], te['labels']), 4),
           'pc_energy': round(te['loss'], 6),
           'spike_rate': round(spk['total'], 6),
           'sps': round(spk['spikes_per_sample'], 2),
           'best_epoch': stopper.best_epoch, 'train_time_s': round(elapsed, 1)}
    row['_history'] = hist
    print(f"  -> acc {row['acc']:.2f}  F1 {row['f1']:.2f}  AUC {row['auc']}  "
          f"SR {row['spike_rate']:.5f}  SPS {row['sps']:.1f}  ({row['train_time_s']}s)", flush=True)
    return row


def save_csv(rows, path):
    keys = [k for k in rows[0] if not k.startswith('_')]
    with open(path, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=keys); w.writeheader()
        w.writerows([{k: r[k] for k in keys} for r in rows])
    print('saved ->', path)

def print_table(rows, title, cols=('variant','neuron','encoding','inference',
                                   'acc','f1','auc','pc_energy','spike_rate','sps')):
    print('\n' + '='*96); print(' ', title); print('='*96)
    print(''.join(f'{c:>14}' if c != 'variant' else f'{c:<26}' for c in cols))
    print('-'*96)
    for r in rows:
        print(''.join(f'{str(r[c]):>14}' if c != 'variant' else f'{str(r[c]):<26}' for c in cols))
    print()


## 4. Table A — neuron model / inference mode / encoding

In [10]:
# ── Cell 8: Table A ───────────────────────────────────────────────────────
VARIANTS_A = [
    # (label,                 neuron,   encoding,        eval_mode)
    ('HH+PC',                 'hh',     'poisson',       'pc'),
    ('HH+PC (FF eval)',       'hh',     'poisson',       'ff'),
    ('LIF+PC',                'lif',    'poisson',       'pc'),
    ('LIF+PC (FF eval)',      'lif',    'poisson',       'ff'),
    ('Analog+PC',             'analog', 'poisson',       'pc'),
    ('HH+PC (latency)',       'hh',     'latency_first', 'pc'),
    ('LIF+PC (latency)',      'lif',    'latency_first', 'pc'),
]

rows_A = []
for label, neuron, enc, mode in VARIANTS_A:
    print(f'[SHD-A] {label} ...', flush=True)
    rows_A.append(run_variant(f'SHD_{label}', neuron=neuron, encoding=enc,
                              eval_mode=mode, soft=True, augment=True))

save_csv(rows_A, f'{OUT_DIR}/table_shd_A_core.csv')
print_table(rows_A, 'Table A — SHD core ablation (neuron / inference / encoding)')


[SHD-A] HH+PC ...


/usr/lib/python3.12/contextlib.py:137: UserWarning: CUDA reports that you have 2 available devices, and you have used fork_rng without explicitly specifying which devices are being used. For safety, we initialize *every* CUDA device by default, which can be quite slow if you have a lot of CUDAs. If you know that you are only making use of a few CUDA devices, set the environment variable CUDA_VISIBLE_DEVICES or the 'devices' keyword argument of fork_rng with the set of devices you are actually using. For example, if you are using CPU only, set device.upper()_VISIBLE_DEVICES= or devices=[]; if you are using device 0 only, set CUDA_VISIBLE_DEVICES=0 or devices=[0].  To initialize all devices and suppress this warning, set the 'devices' keyword argument to `range(torch.cuda.device_count())`.
  return next(self.gen)


  -> acc 84.32  F1 82.83  AUC 0.9827  SR 0.05697  SPS 1196.3  (275.5s)
[SHD-A] HH+PC (FF eval) ...
  -> acc 73.37  F1 70.28  AUC 0.956  SR 0.05588  SPS 1173.4  (298.2s)
[SHD-A] LIF+PC ...
  -> acc 82.86  F1 81.66  AUC 0.9814  SR 0.09137  SPS 1918.9  (172.2s)
[SHD-A] LIF+PC (FF eval) ...
  -> acc 53.67  F1 53.07  AUC 0.9534  SR 0.09214  SPS 1934.9  (135.8s)
[SHD-A] Analog+PC ...
  -> acc 51.63  F1 44.54  AUC 0.833  SR 0.00000  SPS 0.0  (35.2s)
[SHD-A] HH+PC (latency) ...
  -> acc 86.26  F1 85.38  AUC 0.9898  SR 0.02957  SPS 620.9  (309.3s)
[SHD-A] LIF+PC (latency) ...
  -> acc 87.06  F1 85.52  AUC 0.9859  SR 0.08078  SPS 1696.3  (180.0s)
saved -> /kaggle/working/ablation_results/table_shd_A_core.csv

  Table A — SHD core ablation (neuron / inference / encoding)
variant                           neuron      encoding     inference           acc            f1           auc     pc_energy    spike_rate           sps
----------------------------------------------------------------------------

## 5. Table B — SHD-specific ablations

In [11]:
# ── Cell 9: Table B (each row changes exactly one factor off the HH+PC base) ──
VARIANTS_B = [
    # (label,                       kwargs)
    ('base (HH+PC, soft, aug)',     dict()),
    ('hard one-hot targets',        dict(soft=False)),
    ('no augmentation',             dict(augment=False)),
    ('tau_ref=2.0 (67 > steps)',    dict(tau_ref=2.0)),
    ('uniform I_bias=2.0',          dict(i_bias=[2.0, 2.0])),
]

rows_B = []
for label, kw in VARIANTS_B:
    print(f'[SHD-B] {label} ...', flush=True)
    rows_B.append(run_variant(f'SHD_{label}', neuron='hh', encoding='poisson',
                              eval_mode='pc', **kw))

save_csv(rows_B, f'{OUT_DIR}/table_shd_B_specific.csv')
print_table(rows_B, 'Table B — SHD-specific ablation (one factor changed per row)',
            cols=('variant','targets','augment','tau_ref','acc','f1','auc',
                  'pc_energy','spike_rate','sps'))


[SHD-B] base (HH+PC, soft, aug) ...
  -> acc 86.66  F1 86.08  AUC 0.9898  SR 0.05590  SPS 1173.9  (346.5s)
[SHD-B] hard one-hot targets ...
  -> acc 84.50  F1 83.72  AUC 0.9892  SR 0.05810  SPS 1220.0  (278.9s)
[SHD-B] no augmentation ...
  -> acc 82.51  F1 82.24  AUC 0.9823  SR 0.05733  SPS 1204.0  (326.5s)
[SHD-B] tau_ref=2.0 (67 > steps) ...
  -> acc 86.62  F1 86.33  AUC 0.991  SR 0.00655  SPS 137.5  (293.1s)
[SHD-B] uniform I_bias=2.0 ...
  -> acc 85.42  F1 85.46  AUC 0.9894  SR 0.06582  SPS 1382.3  (250.0s)
saved -> /kaggle/working/ablation_results/table_shd_B_specific.csv

  Table B — SHD-specific ablation (one factor changed per row)
variant                          targets       augment       tau_ref           acc            f1           auc     pc_energy    spike_rate           sps
------------------------------------------------------------------------------------------------
SHD_base (HH+PC, soft, aug)          soft             1          0.15         86.66         86.08    

## 6. Spike economy: HH vs LIF at matched accuracy

In [12]:
# ── Cell 10: HH/LIF spike-economy comparison ──────────────────────────────
def _get(rows, name):
    return next(r for r in rows if r['variant'].endswith(name))

hh, lif = _get(rows_A, 'HH+PC'), _get(rows_A, 'LIF+PC')
print(f"{'':22s}{'acc%':>8}{'F1%':>8}{'rate':>10}{'spikes/sample':>16}")
print('-'*64)
for tag, r in (('HH', hh), ('LIF', lif)):
    print(f"{tag:22s}{r['acc']:8.2f}{r['f1']:8.2f}{r['spike_rate']:10.5f}{r['sps']:16.1f}")
red_rate = (lif['spike_rate'] - hh['spike_rate']) / max(lif['spike_rate'], 1e-9) * 100
red_sps  = (lif['sps'] - hh['sps']) / max(lif['sps'], 1e-9) * 100
print(f"\nHH fires {red_rate:+.1f}% of LIF's rate delta "
      f"({red_sps:+.1f}% on spikes/sample); acc delta {hh['acc']-lif['acc']:+.2f} pts")


                          acc%     F1%      rate   spikes/sample
----------------------------------------------------------------
HH                       84.32   82.83   0.05697          1196.3
LIF                      82.86   81.66   0.09137          1918.9

HH fires +37.7% of LIF's rate delta (+37.7% on spikes/sample); acc delta +1.46 pts


## 7. Persist

In [13]:
# ── Cell 11: write summary + JSON ─────────────────────────────────────────
ALL = {'Table A — core (neuron / inference / encoding)': rows_A,
       'Table B — SHD-specific': rows_B}

txt = f'{OUT_DIR}/shd_ablation_summary.txt'
KEYS = ['variant','neuron','encoding','inference','targets','augment','tau_ref',
        'acc','f1','precision','recall','auc','pc_energy','spike_rate','sps',
        'best_epoch','train_time_s']
with open(txt, 'w') as f:
    f.write('HH-PC SNN — SHD ABLATION STUDY\n' + '='*80 + '\n\n')
    f.write(f'config: D_in={D_IN} hidden={HIDDEN} steps_spk={STEPS_SPK} '
            f'T_train={T_TRAIN} T_eval={T_EVAL} epochs<={EPOCHS} seed={SEED}\n'
            f'val speakers held out: {val_spk.tolist()}\n\n')
    for title, rows in ALL.items():
        f.write(title + '\n' + '-'*80 + '\n')
        f.write('  '.join(f'{k:>13}' for k in KEYS) + '\n')
        for r in rows:
            f.write('  '.join(f'{str(r.get(k,""))[:13]:>13}' for k in KEYS) + '\n')
        f.write('\n')

with open(f'{OUT_DIR}/shd_ablation_full.json', 'w') as f:
    json.dump({'config': {'D_in': D_IN, 'hidden': HIDDEN, 'tbins': TBINS, 'pool': POOL,
                          'steps_spk': STEPS_SPK, 'T_train': T_TRAIN, 'T_eval': T_EVAL,
                          'tau_ref': TAU_REF, 'I_bias': I_BIAS, 'lr': LR, 'wd': WD,
                          'epochs': EPOCHS, 'seed': SEED,
                          'val_speakers': val_spk.tolist()},
               'table_A': rows_A, 'table_B': rows_B}, f, indent=2)

print('saved ->', txt)
print('saved ->', f'{OUT_DIR}/shd_ablation_full.json')
print('\nAll SHD ablations complete.')


saved -> /kaggle/working/ablation_results/shd_ablation_summary.txt
saved -> /kaggle/working/ablation_results/shd_ablation_full.json

All SHD ablations complete.
